# Constrained n=200 **animation frames** — A100, saved to Google Drive

Generates the dynamic-Powell **animation frames** (`--frames-all`) for the constrained
lumped n=200 design and writes everything to **Google Drive**, so a Colab disconnect/reset
cannot lose work. Each finished storm's `.bin` files are the checkpoint on Drive; if the
runtime dies, re-run the setup + run cells and it **resumes** (skips storms whose frames
already exist).

**This continues the SAME Drive working dir as the peaks run** (`/content/drive/MyDrive/
mm_dynamic`). That dir already holds the peak checkpoints + `powell_dyn_constrained*.json`
products; the frames are *clamped* to those stored peaks, so they must be present (they are,
from the peaks run — and the uploaded `colab_bundle.zip` now also carries the 4 products as
a fallback for a fresh dir).

Output: `outputs/web/dyn_frames_constrained/` (200 storms x 4 products x 60 KB `.bin`) +
the `dyn_frames_constrained.json` manifest.

**Before running:** `Runtime -> Change runtime type -> A100 GPU`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to A100')

## 2. Mount Drive & set up / resume the working dir
First run: pick `colab_bundle.zip` when prompted. Reconnecting later (or continuing from the
peaks run): it detects the existing Drive copy and **resumes** — no re-upload.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
if os.path.exists(os.path.join(MM, 'pipeline', 'windfield_dynamic_batch.py')):
    print('Working dir already on Drive — will RESUME:', MM)
    fdir = os.path.join(MM, 'outputs/web/dyn_frames_constrained')
    nb = len([f for f in os.listdir(fdir) if f.endswith('.bin')]) if os.path.exists(fdir) else 0
    print(f'  {nb} frame .bin file(s) already on Drive (target = 200 storms x 4 products = 800).')
    prods = [f for f in os.listdir(os.path.join(MM,'outputs/web'))
             if f.startswith('powell_dyn') and f.endswith('_constrained.json')]
    print(f'  stored-peak products present: {sorted(prods)}')
else:
    os.makedirs(MM, exist_ok=True)
    from google.colab import files
    up = files.upload()  # choose colab_bundle.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall(MM)
    print('bundle extracted to Drive:', MM)

## 3. Choose what to run
`--frames-all` re-runs the dynamic B/C/D marches (the expensive ones) for every selected
storm, because their *frames* don't exist yet — so cost ≈ the peaks run (~2 h on the A100).
It is fully resumable: storms whose 4 `.bin` files already exist are skipped.

- `'all'` — all 200 storms (cost-sorted, cheap first)
- `'bulk'` — the cheap 155 · `'tail'` — the expensive 45 · `'vec:75,90'` — specific storms

`BATCH_SIZE`: storms marched together. 25 is safe; on the A100's 40 GB try 100–155 with
`'bulk'`/`'tail'` (keep batches cost-homogeneous).

In [ ]:
SELECT = 'all'      # '' | 'all' | 'bulk' | 'tail' | 'vec:75,90'
BATCH_SIZE = 25     # 25 default; try 100-155 on A100 with SELECT='bulk'

## 4. Run the frames build (writes `.bin` to Drive as it goes)
Streams progress. **If it disconnects:** reconnect, re-run cells 1–2, then this cell — it
skips storms whose frames already exist and resumes.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import subprocess, sys
cmd = [sys.executable, '-u', 'pipeline/windfield_dynamic_batch.py',
       '--constrained', '--frames-all', '--select', SELECT, '--batch-size', str(BATCH_SIZE)]
print('running:', ' '.join(cmd), '\n')
p = subprocess.Popen(cmd, cwd=MM, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if not any(w in line for w in ('Warning','warnings.warn','ConvergenceWarning')):
        print(line, end='')
p.wait()
print('\nexit code:', p.returncode)

## 5. Package the frames + manifest (to Drive **and** download)
Safe to run anytime — it (re)writes the manifest from whatever `.bin` exist (partial runs
OK, no re-compute, no product-clobbering `assemble`), zips the frames dir + manifest to
Drive, and downloads it. Unzip into the repo's `outputs/web/` locally.

In [ ]:
MM = '/content/drive/MyDrive/mm_dynamic'
import sys, os, glob, zipfile
sys.path.insert(0, os.path.join(MM, 'pipeline'))
import windfield_dynamic_batch as B
# point the module's frame globals at the constrained design, then rebuild the manifest
# from the existing .bin (this does NOT run assemble(), so the stored-peak products are safe)
B.FRAMES_DIR = os.path.join(B.WEB, 'dyn_frames_constrained')
B.MANIFEST_JSON = os.path.join(B.WEB, 'dyn_frames_constrained.json')
B.PRODUCTS = {k: (fn.replace('.json', '_constrained.json'), note)
             for k, (fn, note) in B.PRODUCTS.items()}
B.write_frame_manifest()
fdir = os.path.join(MM, 'outputs/web/dyn_frames_constrained')
man  = os.path.join(MM, 'outputs/web/dyn_frames_constrained.json')
bins = sorted(glob.glob(os.path.join(fdir, '*.bin')))
out  = '/content/drive/MyDrive/dynamic_frames_constrained.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(man, os.path.relpath(man, MM))
    for f in bins:
        z.write(f, os.path.relpath(f, MM))
print(f'{len(bins)} frame files + manifest -> {out} (also on Drive)')
from google.colab import files
files.download(out)

### If the runtime disconnects while you're away
Nothing is lost — finished storms' `.bin` are on Drive. On return: re-run **Cell 1**,
**Cell 2** (resumes, no upload), then **Cell 4** to finish, or **Cell 5** to grab what's done.
`dynamic_frames_constrained.zip` also stays on Drive.